In [ ]:
"""
    Breif step-by-step process for simple ANN Workflow:-

        1. Initially import the pandas,numpy, tensorflow, Models-selection, text-preprocessing techinque and other things
        2. Then read the data using pandas code-script using the existing csv data file
        3. Drop irrelevant data from the data set(Cleaning Data) and Label any category coloum and transform it with fit_transform(data['Gender']) like this
        4. Choose another category of clm and convert them into vector form using "OneHotEncoder" or any other txt-preprocess (Data Conversion).
        5. Modify the Dataframe(df) with the latest chenge of "OneHotEncoder" values in the original df and drop the exist df of geography
        6. COnvert them into pickle file using pickle.dump(file-name, file)
        7. DiVide the dataset into indepent and dependent features
        8. ANN Implementation (Import the models, layers and callbacks)
        9. Train the model with Sequential
        10. Next add the optimizer(Adam), loss(binary_crossentropy) and then compile the model. refer:- model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
        11. Once complied then store the logs in a seperate file and add the Earlystoping for it
        12. Train the model with X_train, X_test and Y_train, Y_test refer:- model.fit(
            X_train, y_train, validation_data = (X_test, y_test), epochs = 100,
            callbacks = [tensorflow_callback, early_stopping_callback]
            ) 
        13. Save the model and then check in the tensorflow dashboard how it's working.

"""



import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [ ]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

In [ ]:
# Drop irrelevant data from the data set
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
data

In [ ]:
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
data

In [ ]:
from sklearn.preprocessing import OneHotEncoder
one_hot_encoder_geo = OneHotEncoder()
geo_encoder = one_hot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoder

In [ ]:
one_hot_encoder_geo.get_feature_names_out(['Geography'])

In [ ]:
geo_encoder_df = pd.DataFrame(geo_encoder, columns=one_hot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoder_df

In [ ]:
data = pd.concat([data.drop('Geography', axis=1), geo_encoder_df], axis=1)
data

In [ ]:
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('one_hot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(one_hot_encoder_geo, file)


In [ ]:
## DiVide the dataset into indepent and dependent features
X = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scale = StandardScaler()
X_train = scale.fit_transform(X_train)
X_test = scale.fit_transform(X_test)

In [ ]:
X_train

In [ ]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scale, file)

In [ ]:
data

In [ ]:
# ANN Implementation

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime



In [ ]:
(X_train.shape[1],)

In [ ]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [ ]:
model.summary()

In [ ]:
import tensorflow
opt = tensorflow.keras.optimizers.Adam(learning_rate = 0.1)
loss = tensorflow.keras.losses.BinaryCrossentropy()
loss

In [ ]:
model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
log_dir="logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [ ]:
early_stopping_callback = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)


In [ ]:
# Train The Model
history = model.fit(
    X_train, y_train, validation_data = (X_test, y_test), epochs = 100,
    callbacks = [tensorflow_callback, early_stopping_callback]
) 

In [ ]:
model.save('model.h5')

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir logs/fit/